# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


In [1]:
## Question 1 Solution
class Matrix:
    def __init__(self, *args):
        if len(args) == 2 and isinstance(args[0], int) and isinstance(args[1], int):
            n, m = args
            if n <= 0 or m <= 0:
                raise ValueError("Matrix dimensions n and m must be positive integers.")
            self.n = n
            self.m = m
            self.data = [[0 for _ in range(m)] for _ in range(n)]

        elif len(args) == 1 and isinstance(args[0], list):
            grid = args[0]
            self._validate_list_of_lists(grid)
            self.n = len(grid)
            self.m = len(grid[0])
            self.data = [list(row) for row in grid]

        else:
            raise TypeError(
                "Initialize Matrix with either (n, m) integers or a single 2D list of lists."
            )

    @staticmethod
    def _validate_list_of_lists(grid: list) -> None:
        if not isinstance(grid, list) or len(grid) == 0:
            raise ValueError("Matrix must be a non-empty list of lists.")
        if not all(isinstance(row, list) for row in grid):
            raise ValueError("All rows in the matrix must be lists.")

        num_cols = len(grid[0])
        if num_cols == 0:
            raise ValueError("Matrix rows cannot be empty.")
        if not all(len(row) == num_cols for row in grid):
            raise ValueError("Invalid matrix: all rows must have the same number of columns.")

    def shape(self) -> tuple[int, int]:
        """Returns the (n, m) dimensions of the matrix."""
        return (self.n, self.m)

    def __getitem__(self, index):
        if isinstance(index, int):
            return self.data[index]
        elif isinstance(index, tuple) and len(index) == 2:
            i, j = index
            if isinstance(i, int) and isinstance(j, int):
                return self.data[i][j]
            elif isinstance(i, slice) and isinstance(j, slice):
                return [row[j] for row in self.data[i]]
            else:
                raise TypeError("Tuple indices must be two integers or two slices.")
        elif isinstance(index, slice):
            return self.data[index]
        else:
            raise TypeError("Invalid index type.")

    def __setitem__(self, index, value):
        if isinstance(index, tuple) and len(index) == 2:
            i, j = index
            if isinstance(i, int) and isinstance(j, int):
                self.data[i][j] = value
            elif isinstance(i, slice) and isinstance(j, slice):
                self.assign(value)
            else:
                raise TypeError("Invalid tuple index for assignment.")
        elif isinstance(index, slice) and index == slice(None):
            self.assign(value)
        elif isinstance(index, int):
            if not isinstance(value, list) or len(value) != self.m:
                raise ValueError(f"Assigned row must be a list of length {self.m}.")
            self.data[index] = list(value)
        else:
            raise TypeError("Invalid index type for assignment.")

    def assign(self, other) -> "Matrix":
        if isinstance(other, Matrix):
            if (self.n, self.m) != (other.n, other.m):
                raise ValueError(
                    f"Dimension mismatch: cannot assign {other.n}x{other.m} Matrix "
                    f"to {self.n}x{self.m} Matrix."
                )
            self.data = [list(row) for row in other.data]

        elif isinstance(other, list):
            self._validate_list_of_lists(other)
            if (self.n, self.m) != (len(other), len(other[0])):
                raise ValueError(
                    f"Dimension mismatch: cannot assign {len(other)}x{len(other[0])} "
                    f"list of lists to {self.n}x{self.m} Matrix."
                )
            self.data = [list(row) for row in other]

        else:
            raise TypeError("Can only assign from another Matrix or a 2D list of lists.")

        return self

    def __repr__(self) -> str:
        rows_str = ",\n       ".join(repr(row) for row in self.data)
        return f"Matrix([{rows_str}])"




2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

In [2]:
#Question 2 Solution
class Matrix:
    def __init__(self, *args):

        if len(args) == 2 and isinstance(args[0], int) and isinstance(args[1], int):
            n, m = args
            if n <= 0 or m <= 0:
                raise ValueError("Matrix dimensions n and m must be positive integers.")
            self.n = n
            self.m = m
            self.data = [[0 for _ in range(m)] for _ in range(n)]

        elif len(args) == 1 and isinstance(args[0], list):
            grid = args[0]
            self._validate_list_of_lists(grid)
            self.n = len(grid)
            self.m = len(grid[0])
            self.data = [list(row) for row in grid]

        else:
            raise TypeError(
                "Initialize Matrix with either (n, m) integers or a single 2D list of lists."
            )

    @staticmethod
    def _validate_list_of_lists(grid: list) -> None:
        if not isinstance(grid, list) or len(grid) == 0:
            raise ValueError("Matrix must be a non-empty list of lists.")
        if not all(isinstance(row, list) for row in grid):
            raise ValueError("All rows in the matrix must be lists.")

        num_cols = len(grid[0])
        if num_cols == 0:
            raise ValueError("Matrix rows cannot be empty.")
        if not all(len(row) == num_cols for row in grid):
            raise ValueError("Invalid matrix: all rows must have the same number of columns.")

    def shape(self) -> tuple[int, int]:
        return (self.n, self.m)

    def transpose(self) -> "Matrix":
        transposed_data = [
            [self.data[i][j] for i in range(self.n)] for j in range(self.m)
        ]
        return Matrix(transposed_data)

    def row(self, n: int) -> "Matrix":
        if not (-self.n <= n < self.n):
            raise IndexError(f"Row index {n} out of bounds for matrix with {self.n} rows.")
        return Matrix([list(self.data[n])])

    def column(self, n: int) -> "Matrix":
        if not (-self.m <= n < self.m):
            raise IndexError(f"Column index {n} out of bounds for matrix with {self.m} columns.")
        return Matrix([[self.data[i][n]] for i in range(self.n)])

    def to_list(self) -> list[list]:
        return [list(row) for row in self.data]

    def block(self, n_0: int, n_1: int, m_0: int, m_1: int) -> "Matrix":

        sub_rows = self.data[m_0:m_1]
        sub_grid = [row[n_0:n_1] for row in sub_rows]
        if not sub_grid or not sub_grid[0]:
            raise ValueError("The specified block indices resulted in an empty sub-matrix.")
        return Matrix(sub_grid)

    def __getitem__(self, index):
        if isinstance(index, int):
            return self.data[index]

        elif isinstance(index, slice):
            sliced_rows = [list(row) for row in self.data[index]]
            if not sliced_rows:
                raise ValueError("Slice resulted in an empty matrix.")
            return Matrix(sliced_rows)

        elif isinstance(index, tuple) and len(index) == 2:
            row_idx, col_idx = index

            if isinstance(row_idx, int) and isinstance(col_idx, int):
                return self.data[row_idx][col_idx]

            elif isinstance(row_idx, int) and isinstance(col_idx, slice):
                sub_row = self.data[row_idx][col_idx]
                if not sub_row:
                    raise ValueError("Slice resulted in an empty matrix.")
                return Matrix([list(sub_row)])

            elif isinstance(row_idx, slice) and isinstance(col_idx, int):
                sub_col = [[row[col_idx]] for row in self.data[row_idx]]
                if not sub_col:
                    raise ValueError("Slice resulted in an empty matrix.")
                return Matrix(sub_col)

            elif isinstance(row_idx, slice) and isinstance(col_idx, slice):
                sub_grid = [list(row[col_idx]) for row in self.data[row_idx]]
                if not sub_grid or not sub_grid[0]:
                    raise ValueError("Slice resulted in an empty matrix.")
                return Matrix(sub_grid)

            else:
                raise TypeError("Tuple indices must be integers or slices.")

        else:
            raise TypeError("Invalid index type.")

    def __setitem__(self, index, value):
        if isinstance(index, tuple) and len(index) == 2:
            i, j = index
            if isinstance(i, int) and isinstance(j, int):
                self.data[i][j] = value
            elif isinstance(i, slice) and isinstance(j, slice):
                if i == slice(None) and j == slice(None):
                    self.assign(value)
                else:
                    target_rows = range(*i.indices(self.n))
                    target_cols = range(*j.indices(self.m))
                    val_matrix = value if isinstance(value, Matrix) else Matrix(value)
                    if val_matrix.shape() != (len(target_rows), len(target_cols)):
                        raise ValueError("Slice assignment dimensions do not match.")
                    for r_idx, r in enumerate(target_rows):
                        for c_idx, c in enumerate(target_cols):
                            self.data[r][c] = val_matrix[r_idx, c_idx]
            else:
                raise TypeError("Invalid tuple index for assignment.")
        elif isinstance(index, slice) and index == slice(None):
            self.assign(value)
        elif isinstance(index, int):
            if not isinstance(value, list) or len(value) != self.m:
                raise ValueError(f"Assigned row must be a list of length {self.m}.")
            self.data[index] = list(value)
        else:
            raise TypeError("Invalid index type for assignment.")

    def assign(self, other) -> "Matrix":
        if isinstance(other, Matrix):
            if (self.n, self.m) != (other.n, other.m):
                raise ValueError(
                    f"Dimension mismatch: cannot assign {other.n}x{other.m} Matrix "
                    f"to {self.n}x{self.m} Matrix."
                )
            self.data = [list(row) for row in other.data]

        elif isinstance(other, list):
            self._validate_list_of_lists(other)
            if (self.n, self.m) != (len(other), len(other[0])):
                raise ValueError(
                    f"Dimension mismatch: cannot assign {len(other)}x{len(other[0])} "
                    f"list of lists to {self.n}x{self.m} Matrix."
                )
            self.data = [list(row) for row in other]

        else:
            raise TypeError("Can only assign from another Matrix or a 2D list of lists.")

        return self

    def __repr__(self) -> str:
        rows_str = ",\n       ".join(repr(row) for row in self.data)
        return f"Matrix([{rows_str}])"

In [3]:
#Question 2 Test
if __name__ == "__main__":
    M = Matrix([
        [ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12]
    ])
    print("Original Matrix M:")
    print(M)

    print(f"\n1. M.shape(): {M.shape()}")

    print("\n2. M.transpose() (shape 4x3):")
    print(M.transpose())

    print("\n3a. M.row(1) (1x4 Matrix):")
    print(M.row(1))
    print("\n3b. M.column(2) (3x1 Matrix):")
    print(M.column(2))

    print(f"\n4. M.to_list(): {M.to_list()}")

    print("\n5. M.block(1, 3, 0, 2) (columns 1..3, rows 0..2):")
    print(M.block(1, 3, 0, 2))

    print("\n6a. Slicing rows and columns M[0:2, 1:4]:")
    print(M[0:2, 1:4])

    print("\n6b. Slicing with step M[:, ::2] (all rows, every 2nd column):")
    print(M[:, ::2])

    print("\n6c. Single row slice M[1, 1:3]:")
    print(M[1, 1:3])

Original Matrix M:
Matrix([[1, 2, 3, 4],
       [5, 6, 7, 8],
       [9, 10, 11, 12]])

1. M.shape(): (3, 4)

2. M.transpose() (shape 4x3):
Matrix([[1, 5, 9],
       [2, 6, 10],
       [3, 7, 11],
       [4, 8, 12]])

3a. M.row(1) (1x4 Matrix):
Matrix([[5, 6, 7, 8]])

3b. M.column(2) (3x1 Matrix):
Matrix([[3],
       [7],
       [11]])

4. M.to_list(): [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]

5. M.block(1, 3, 0, 2) (columns 1..3, rows 0..2):
Matrix([[2, 3],
       [6, 7]])

6a. Slicing rows and columns M[0:2, 1:4]:
Matrix([[2, 3, 4],
       [6, 7, 8]])

6b. Slicing with step M[:, ::2] (all rows, every 2nd column):
Matrix([[1, 3],
       [5, 7],
       [9, 11]])

6c. Single row slice M[1, 1:3]:
Matrix([[6, 7]])


3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

In [4]:
#Question 3 Solution

def constant(n: int, m: int, c: float) -> Matrix:
    if n <= 0 or m <= 0:
        raise ValueError("Matrix dimensions n and m must be positive integers.")
    val = float(c)
    return Matrix([[val for _ in range(m)] for _ in range(n)])

def zeros(n: int, m: int) -> Matrix:
    return constant(n, m, 0.0)

def ones(n: int, m: int) -> Matrix:
    return constant(n, m, 1.0)

def eye(n: int) -> Matrix:
    if n <= 0:
        raise ValueError("Dimension n must be a positive integer.")
    M = zeros(n, n)
    for i in range(n):
        M[i, i] = 1.0
    return M

In [5]:
#Question 3 Test

if __name__ == "__main__":
    print("1. constant(2, 3, 7):")
    print(constant(2, 3, 7))

    print("\n2. zeros(2, 4):")
    print(zeros(2, 4))

    print("\n3. ones(3, 2):")
    print(ones(3, 2))

    print("\n4. eye(3):")
    print(eye(3))

1. constant(2, 3, 7):
Matrix([[7.0, 7.0, 7.0],
       [7.0, 7.0, 7.0]])

2. zeros(2, 4):
Matrix([[0.0, 0.0, 0.0, 0.0],
       [0.0, 0.0, 0.0, 0.0]])

3. ones(3, 2):
Matrix([[1.0, 1.0],
       [1.0, 1.0],
       [1.0, 1.0]])

4. eye(3):
Matrix([[1.0, 0.0, 0.0],
       [0.0, 1.0, 0.0],
       [0.0, 0.0, 1.0]])


4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

In [6]:
#Question 4 Solution
class Matrix:
    def __init__(self, *args):

        if len(args) == 2 and isinstance(args[0], int) and isinstance(args[1], int):
            n, m = args
            if n <= 0 or m <= 0:
                raise ValueError("Matrix dimensions n and m must be positive integers.")
            self.n = n
            self.m = m
            self.data = [[0 for _ in range(m)] for _ in range(n)]

        elif len(args) == 1 and isinstance(args[0], list):
            grid = args[0]
            self._validate_list_of_lists(grid)
            self.n = len(grid)
            self.m = len(grid[0])
            self.data = [list(row) for row in grid]

        else:
            raise TypeError(
                "Initialize Matrix with either (n, m) integers or a single 2D list of lists."
            )

    @staticmethod
    def _validate_list_of_lists(grid: list) -> None:
        if not isinstance(grid, list) or len(grid) == 0:
            raise ValueError("Matrix must be a non-empty list of lists.")
        if not all(isinstance(row, list) for row in grid):
            raise ValueError("All rows in the matrix must be lists.")

        num_cols = len(grid[0])
        if num_cols == 0:
            raise ValueError("Matrix rows cannot be empty.")
        if not all(len(row) == num_cols for row in grid):
            raise ValueError("Invalid matrix: all rows must have the same number of columns.")

    def shape(self) -> tuple[int, int]:
        return (self.n, self.m)

    def transpose(self) -> "Matrix":
        transposed_data = [
            [self.data[i][j] for i in range(self.n)] for j in range(self.m)
        ]
        return Matrix(transposed_data)

    def row(self, n: int) -> "Matrix":
        if not (-self.n <= n < self.n):
            raise IndexError(f"Row index {n} out of bounds for matrix with {self.n} rows.")
        return Matrix([list(self.data[n])])

    def column(self, n: int) -> "Matrix":
        if not (-self.m <= n < self.m):
            raise IndexError(f"Column index {n} out of bounds for matrix with {self.m} columns.")
        return Matrix([[self.data[i][n]] for i in range(self.n)])

    def to_list(self) -> list[list]:
        return [list(row) for row in self.data]

    def block(self, n_0: int, n_1: int, m_0: int, m_1: int) -> "Matrix":
        sub_rows = self.data[m_0:m_1]
        sub_grid = [row[n_0:n_1] for row in sub_rows]
        if not sub_grid or not sub_grid[0]:
            raise ValueError("The specified block indices resulted in an empty sub-matrix.")
        return Matrix(sub_grid)

    def _ensure_same_shape(self, other: "Matrix", op_name: str) -> None:
        if not isinstance(other, Matrix):
            raise TypeError(f"{op_name} requires another Matrix instance.")
        if self.shape() != other.shape():
            raise ValueError(
                f"Dimension mismatch for {op_name}: {self.shape()} vs {other.shape()}."
            )

    def scalarmul(self, c: float | int) -> "Matrix":
        if not isinstance(c, (int, float)):
            raise TypeError("Scalar multiplier c must be an int or float.")
        return Matrix([[c * self.data[i][j] for j in range(self.m)] for i in range(self.n)])

    def add(self, N: "Matrix") -> "Matrix":
        self._ensure_same_shape(N, "add")
        return Matrix([
            [self.data[i][j] + N.data[i][j] for j in range(self.m)]
            for i in range(self.n)
        ])

    def sub(self, N: "Matrix") -> "Matrix":
        self._ensure_same_shape(N, "sub")
        return Matrix([
            [self.data[i][j] - N.data[i][j] for j in range(self.m)]
            for i in range(self.n)
        ])

    def element_mult(self, N: "Matrix") -> "Matrix":
        self._ensure_same_shape(N, "element_mult")
        return Matrix([
            [self.data[i][j] * N.data[i][j] for j in range(self.m)]
            for i in range(self.n)
        ])

    def mat_mult(self, N: "Matrix") -> "Matrix":

        if not isinstance(N, Matrix):
            raise TypeError("mat_mult requires another Matrix instance.")
        if self.m != N.n:
            raise ValueError(
                f"Incompatible dimensions for matrix multiplication: "
                f"{self.shape()} cannot be multiplied by {N.shape()} "
                f"(columns of M ({self.m}) must equal rows of N ({N.n}))."
            )
        result = [
            [
                sum(self.data[i][k] * N.data[k][j] for k in range(self.m))
                for j in range(N.m)
            ]
            for i in range(self.n)
        ]
        return Matrix(result)

    def equals(self, N: object) -> bool:
        if not isinstance(N, Matrix):
            return False
        if self.shape() != N.shape():
            return False
        return all(
            self.data[i][j] == N.data[i][j]
            for i in range(self.n)
            for j in range(self.m)
        )

    def __getitem__(self, index):
        if isinstance(index, int):
            return self.data[index]
        elif isinstance(index, slice):
            sliced_rows = [list(row) for row in self.data[index]]
            if not sliced_rows:
                raise ValueError("Slice resulted in an empty matrix.")
            return Matrix(sliced_rows)
        elif isinstance(index, tuple) and len(index) == 2:
            row_idx, col_idx = index
            if isinstance(row_idx, int) and isinstance(col_idx, int):
                return self.data[row_idx][col_idx]
            elif isinstance(row_idx, int) and isinstance(col_idx, slice):
                sub_row = self.data[row_idx][col_idx]
                if not sub_row:
                    raise ValueError("Slice resulted in an empty matrix.")
                return Matrix([list(sub_row)])
            elif isinstance(row_idx, slice) and isinstance(col_idx, int):
                sub_col = [[row[col_idx]] for row in self.data[row_idx]]
                if not sub_col:
                    raise ValueError("Slice resulted in an empty matrix.")
                return Matrix(sub_col)
            elif isinstance(row_idx, slice) and isinstance(col_idx, slice):
                sub_grid = [list(row[col_idx]) for row in self.data[row_idx]]
                if not sub_grid or not sub_grid[0]:
                    raise ValueError("Slice resulted in an empty matrix.")
                return Matrix(sub_grid)
            else:
                raise TypeError("Tuple indices must be integers or slices.")
        else:
            raise TypeError("Invalid index type.")

    def __setitem__(self, index, value):
        if isinstance(index, tuple) and len(index) == 2:
            i, j = index
            if isinstance(i, int) and isinstance(j, int):
                self.data[i][j] = value
            elif isinstance(i, slice) and isinstance(j, slice):
                if i == slice(None) and j == slice(None):
                    self.assign(value)
                else:
                    target_rows = range(*i.indices(self.n))
                    target_cols = range(*j.indices(self.m))
                    val_matrix = value if isinstance(value, Matrix) else Matrix(value)
                    if val_matrix.shape() != (len(target_rows), len(target_cols)):
                        raise ValueError("Slice assignment dimensions do not match.")
                    for r_idx, r in enumerate(target_rows):
                        for c_idx, c in enumerate(target_cols):
                            self.data[r][c] = val_matrix[r_idx, c_idx]
            else:
                raise TypeError("Invalid tuple index for assignment.")
        elif isinstance(index, slice) and index == slice(None):
            self.assign(value)
        elif isinstance(index, int):
            if not isinstance(value, list) or len(value) != self.m:
                raise ValueError(f"Assigned row must be a list of length {self.m}.")
            self.data[index] = list(value)
        else:
            raise TypeError("Invalid index type for assignment.")

    def assign(self, other) -> "Matrix":
        if isinstance(other, Matrix):
            if (self.n, self.m) != (other.n, other.m):
                raise ValueError(
                    f"Dimension mismatch: cannot assign {other.n}x{other.m} Matrix "
                    f"to {self.n}x{self.m} Matrix."
                )
            self.data = [list(row) for row in other.data]
        elif isinstance(other, list):
            self._validate_list_of_lists(other)
            if (self.n, self.m) != (len(other), len(other[0])):
                raise ValueError(
                    f"Dimension mismatch: cannot assign {len(other)}x{len(other[0])} "
                    f"list of lists to {self.n}x{self.m} Matrix."
                )
            self.data = [list(row) for row in other]
        else:
            raise TypeError("Can only assign from another Matrix or a 2D list of lists.")
        return self

    def __repr__(self) -> str:
        rows_str = ",\n       ".join(repr(row) for row in self.data)
        return f"Matrix([{rows_str}])"


In [7]:
#Question 4 Test
if __name__ == "__main__":
    M = Matrix([[1, 2, 3], [4, 5, 6]])
    N = Matrix([[10, 20, 30], [40, 50, 60]])
    P = Matrix([[1, 2], [3, 4], [5, 6]])
    print("1. M.scalarmul(3):")
    print(M.scalarmul(3))

    print("\n2. M.add(N):")
    print(M.add(N))

    print("\n3. N.sub(M):")
    print(N.sub(M))

    print("\n4. M.element_mult(N) (2x3 * 2x3 element-wise):")
    print(M.element_mult(N))

    print("\n5. M.mat_mult(P) (2x3 @ 3x2 -> 2x2):")
    print(M.mat_mult(P))

    print(f"\n6. M.equals(N): {M.equals(N)}")
    print(f"   M.equals(Matrix([[1, 2, 3], [4, 5, 6]])): {M.equals(Matrix([[1, 2, 3], [4, 5, 6]]))}")

    try:
        M.add(P)
    except ValueError as e:
        print(f"\nCaught expected add() error: {e}")

    try:
        M.mat_mult(N)
    except ValueError as e:
        print(f"Caught expected mat_mult() error: {e}")

1. M.scalarmul(3):
Matrix([[3, 6, 9],
       [12, 15, 18]])

2. M.add(N):
Matrix([[11, 22, 33],
       [44, 55, 66]])

3. N.sub(M):
Matrix([[9, 18, 27],
       [36, 45, 54]])

4. M.element_mult(N) (2x3 * 2x3 element-wise):
Matrix([[10, 40, 90],
       [160, 250, 360]])

5. M.mat_mult(P) (2x3 @ 3x2 -> 2x2):
Matrix([[22, 28],
       [49, 64]])

6. M.equals(N): False
   M.equals(Matrix([[1, 2, 3], [4, 5, 6]])): True

Caught expected add() error: Dimension mismatch for add: (2, 3) vs (3, 2).
Caught expected mat_mult() error: Incompatible dimensions for matrix multiplication: (2, 3) cannot be multiplied by (2, 3) (columns of M (3) must equal rows of N (2)).


5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N


In [8]:
#Question 5 Solution
class Matrix:
    def __init__(self, *args):
        if len(args) == 2 and isinstance(args[0], int) and isinstance(args[1], int):
            n, m = args
            if n <= 0 or m <= 0:
                raise ValueError("Matrix dimensions n and m must be positive integers.")
            self.n = n
            self.m = m
            self.data = [[0 for _ in range(m)] for _ in range(n)]

        elif len(args) == 1 and isinstance(args[0], list):
            grid = args[0]
            self._validate_list_of_lists(grid)
            self.n = len(grid)
            self.m = len(grid[0])
            self.data = [list(row) for row in grid]

        else:
            raise TypeError(
                "Initialize Matrix with either (n, m) integers or a single 2D list of lists."
            )

    @staticmethod
    def _validate_list_of_lists(grid: list) -> None:
        if not isinstance(grid, list) or len(grid) == 0:
            raise ValueError("Matrix must be a non-empty list of lists.")
        if not all(isinstance(row, list) for row in grid):
            raise ValueError("All rows in the matrix must be lists.")

        num_cols = len(grid[0])
        if num_cols == 0:
            raise ValueError("Matrix rows cannot be empty.")
        if not all(len(row) == num_cols for row in grid):
            raise ValueError("Invalid matrix: all rows must have the same number of columns.")

    def shape(self) -> tuple[int, int]:
        return (self.n, self.m)

    def transpose(self) -> "Matrix":
        return Matrix([[self.data[i][j] for i in range(self.n)] for j in range(self.m)])

    def row(self, n: int) -> "Matrix":
        if not (-self.n <= n < self.n):
            raise IndexError(f"Row index {n} out of bounds for matrix with {self.n} rows.")
        return Matrix([list(self.data[n])])

    def column(self, n: int) -> "Matrix":
        if not (-self.m <= n < self.m):
            raise IndexError(f"Column index {n} out of bounds for matrix with {self.m} columns.")
        return Matrix([[self.data[i][n]] for i in range(self.n)])

    def to_list(self) -> list[list]:
        return [list(row) for row in self.data]

    def block(self, n_0: int, n_1: int, m_0: int, m_1: int) -> "Matrix":
        sub_rows = self.data[m_0:m_1]
        sub_grid = [row[n_0:n_1] for row in sub_rows]
        if not sub_grid or not sub_grid[0]:
            raise ValueError("The specified block indices resulted in an empty sub-matrix.")
        return Matrix(sub_grid)

    def _ensure_same_shape(self, other: "Matrix", op_name: str) -> None:
        if not isinstance(other, Matrix):
            raise TypeError(f"{op_name} requires another Matrix instance.")
        if self.shape() != other.shape():
            raise ValueError(
                f"Dimension mismatch for {op_name}: {self.shape()} vs {other.shape()}."
            )

    def scalarmul(self, c: float | int) -> "Matrix":
        if not isinstance(c, (int, float)):
            raise TypeError("Scalar multiplier c must be an int or float.")
        return Matrix([[c * self.data[i][j] for j in range(self.m)] for i in range(self.n)])

    def add(self, N: "Matrix") -> "Matrix":
        self._ensure_same_shape(N, "add")
        return Matrix([
            [self.data[i][j] + N.data[i][j] for j in range(self.m)]
            for i in range(self.n)
        ])

    def sub(self, N: "Matrix") -> "Matrix":
        self._ensure_same_shape(N, "sub")
        return Matrix([
            [self.data[i][j] - N.data[i][j] for j in range(self.m)]
            for i in range(self.n)
        ])

    def element_mult(self, N: "Matrix") -> "Matrix":
        self._ensure_same_shape(N, "element_mult")
        return Matrix([
            [self.data[i][j] * N.data[i][j] for j in range(self.m)]
            for i in range(self.n)
        ])

    def mat_mult(self, N: "Matrix") -> "Matrix":
        if not isinstance(N, Matrix):
            raise TypeError("mat_mult requires another Matrix instance.")
        if self.m != N.n:
            raise ValueError(
                f"Incompatible dimensions for matrix multiplication: "
                f"{self.shape()} cannot be multiplied by {N.shape()} "
                f"(columns of M ({self.m}) must equal rows of N ({N.n}))."
            )
        result = [
            [
                sum(self.data[i][k] * N.data[k][j] for k in range(self.m))
                for j in range(N.m)
            ]
            for i in range(self.n)
        ]
        return Matrix(result)

    def equals(self, N: object) -> bool:
        if not isinstance(N, Matrix):
            return False
        if self.shape() != N.shape():
            return False
        return all(
            self.data[i][j] == N.data[i][j]
            for i in range(self.n)
            for j in range(self.m)
        )

    def __add__(self, other: "Matrix") -> "Matrix":
        return self.add(other)

    def __sub__(self, other: "Matrix") -> "Matrix":
        return self.sub(other)

    def __mul__(self, other: "Matrix | float | int") -> "Matrix":

        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        elif isinstance(other, Matrix):
            return self.mat_mult(other)
        return NotImplemented

    def __rmul__(self, other: float | int) -> "Matrix":
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        return NotImplemented

    def __matmul__(self, other: "Matrix") -> "Matrix":
        if isinstance(other, Matrix):
            return self.mat_mult(other)
        return NotImplemented

    def __eq__(self, other: object) -> bool:
        return self.equals(other)

    def __getitem__(self, index):
        if isinstance(index, int):
            return self.data[index]
        elif isinstance(index, slice):
            sliced_rows = [list(row) for row in self.data[index]]
            if not sliced_rows:
                raise ValueError("Slice resulted in an empty matrix.")
            return Matrix(sliced_rows)
        elif isinstance(index, tuple) and len(index) == 2:
            row_idx, col_idx = index
            if isinstance(row_idx, int) and isinstance(col_idx, int):
                return self.data[row_idx][col_idx]
            elif isinstance(row_idx, int) and isinstance(col_idx, slice):
                sub_row = self.data[row_idx][col_idx]
                if not sub_row:
                    raise ValueError("Slice resulted in an empty matrix.")
                return Matrix([list(sub_row)])
            elif isinstance(row_idx, slice) and isinstance(col_idx, int):
                sub_col = [[row[col_idx]] for row in self.data[row_idx]]
                if not sub_col:
                    raise ValueError("Slice resulted in an empty matrix.")
                return Matrix(sub_col)
            elif isinstance(row_idx, slice) and isinstance(col_idx, slice):
                sub_grid = [list(row[col_idx]) for row in self.data[row_idx]]
                if not sub_grid or not sub_grid[0]:
                    raise ValueError("Slice resulted in an empty matrix.")
                return Matrix(sub_grid)
            else:
                raise TypeError("Tuple indices must be integers or slices.")
        else:
            raise TypeError("Invalid index type.")

    def __setitem__(self, index, value):
        if isinstance(index, tuple) and len(index) == 2:
            i, j = index
            if isinstance(i, int) and isinstance(j, int):
                self.data[i][j] = value
            elif isinstance(i, slice) and isinstance(j, slice):
                if i == slice(None) and j == slice(None):
                    self.assign(value)
                else:
                    target_rows = range(*i.indices(self.n))
                    target_cols = range(*j.indices(self.m))
                    val_matrix = value if isinstance(value, Matrix) else Matrix(value)
                    if val_matrix.shape() != (len(target_rows), len(target_cols)):
                        raise ValueError("Slice assignment dimensions do not match.")
                    for r_idx, r in enumerate(target_rows):
                        for c_idx, c in enumerate(target_cols):
                            self.data[r][c] = val_matrix[r_idx, c_idx]
            else:
                raise TypeError("Invalid tuple index for assignment.")
        elif isinstance(index, slice) and index == slice(None):
            self.assign(value)
        elif isinstance(index, int):
            if not isinstance(value, list) or len(value) != self.m:
                raise ValueError(f"Assigned row must be a list of length {self.m}.")
            self.data[index] = list(value)
        else:
            raise TypeError("Invalid index type for assignment.")

    def assign(self, other) -> "Matrix":
        if isinstance(other, Matrix):
            if (self.n, self.m) != (other.n, other.m):
                raise ValueError(
                    f"Dimension mismatch: cannot assign {other.n}x{other.m} Matrix "
                    f"to {self.n}x{self.m} Matrix."
                )
            self.data = [list(row) for row in other.data]
        elif isinstance(other, list):
            self._validate_list_of_lists(other)
            if (self.n, self.m) != (len(other), len(other[0])):
                raise ValueError(
                    f"Dimension mismatch: cannot assign {len(other)}x{len(other[0])} "
                    f"list of lists to {self.n}x{self.m} Matrix."
                )
            self.data = [list(row) for row in other]
        else:
            raise TypeError("Can only assign from another Matrix or a 2D list of lists.")
        return self

    def __repr__(self) -> str:
        rows_str = ",\n       ".join(repr(row) for row in self.data)
        return f"Matrix([{rows_str}])"




In [9]:
#Question 5 Test
if __name__ == "__main__":
    M = Matrix([[1, 2, 3], [4, 5, 6]])
    N = Matrix([[10, 20, 30], [40, 50, 60]])
    P = Matrix([[1, 2], [3, 4], [5, 6]])
    print("1. 2 * M (__rmul__):")
    print(2 * M)

    print("\n2. M * 2 (__mul__ with scalar):")
    print(M * 2)

    print("\n3. M + N (__add__):")
    print(M + N)

    print("\n4. N - M (__sub__):")
    print(N - M)

    print("\n5. M * P (__mul__ with Matrix -> 2x3 * 3x2 = 2x2):")
    print(M * P)

    print(f"\n6. M == N (__eq__): {M == N}")
    print(f"   2 * M == M * 2:  {2 * M == M * 2}")

1. 2 * M (__rmul__):
Matrix([[2, 4, 6],
       [8, 10, 12]])

2. M * 2 (__mul__ with scalar):
Matrix([[2, 4, 6],
       [8, 10, 12]])

3. M + N (__add__):
Matrix([[11, 22, 33],
       [44, 55, 66]])

4. N - M (__sub__):
Matrix([[9, 18, 27],
       [36, 45, 54]])

5. M * P (__mul__ with Matrix -> 2x3 * 3x2 = 2x2):
Matrix([[22, 28],
       [49, 64]])

6. M == N (__eq__): False
   2 * M == M * 2:  True


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [10]:
#Question 6 Solution
if __name__ == "__main__":
    A = Matrix([[1, 2], [3, 4]])
    B = Matrix([[2, 0], [1, 3]])
    C = Matrix([[1, 4], [2, 1]])
    I = eye(2)

    print("Example 2x2 Matrices:")
    print(f"A =\n{A}\n")
    print(f"B =\n{B}\n")
    print(f"C =\n{C}\n")
    print(f"I = eye(2) =\n{I}\n")

    lhs_1 = (A * B) * C
    rhs_1 = A * (B * C)
    print("1. Associativity: (AB)C = A(BC)")
    print(f"(A * B) * C =\n{lhs_1}")
    print(f"A * (B * C) =\n{rhs_1}")
    print(f"(A * B) * C == A * (B * C) -> {lhs_1 == rhs_1}\n")

    lhs_2 = A * (B + C)
    rhs_2 = (A * B) + (A * C)
    print("2. Distributivity: A(B + C) = AB + AC")
    print(f"A * (B + C) =\n{lhs_2}")
    print(f"(A * B) + (A * C) =\n{rhs_2}")
    print(f"A * (B + C) == (A * B) + (A * C) -> {lhs_2 == rhs_2}\n")

    ab = A * B
    ba = B * A
    print("3. Non-Commutativity: AB != BA")
    print(f"A * B =\n{ab}")
    print(f"B * A =\n{ba}")
    print(f"A * B == B * A -> {ab == ba}")
    print(f"A * B != B * A -> {ab != ba}\n")

    ai = A * I
    print("4. Multiplicative Identity: AI = A")
    print(f"A * I =\n{ai}")
    print(f"A =\n{A}")
    print(f"A * I == A -> {ai == A}")

Example 2x2 Matrices:
A =
Matrix([[1, 2],
       [3, 4]])

B =
Matrix([[2, 0],
       [1, 3]])

C =
Matrix([[1, 4],
       [2, 1]])

I = eye(2) =
Matrix([[1.0, 0.0],
       [0.0, 1.0]])

1. Associativity: (AB)C = A(BC)
(A * B) * C =
Matrix([[16, 22],
       [34, 52]])
A * (B * C) =
Matrix([[16, 22],
       [34, 52]])
(A * B) * C == A * (B * C) -> True

2. Distributivity: A(B + C) = AB + AC
A * (B + C) =
Matrix([[9, 12],
       [21, 28]])
(A * B) + (A * C) =
Matrix([[9, 12],
       [21, 28]])
A * (B + C) == (A * B) + (A * C) -> True

3. Non-Commutativity: AB != BA
A * B =
Matrix([[4, 6],
       [10, 12]])
B * A =
Matrix([[2, 4],
       [10, 14]])
A * B == B * A -> False
A * B != B * A -> True

4. Multiplicative Identity: AI = A
A * I =
Matrix([[1.0, 2.0],
       [3.0, 4.0]])
A =
Matrix([[1, 2],
       [3, 4]])
A * I == A -> True
